In [ ]:

from pathlib import Path
import os
import pandas as pd

# Repository containing the dataset
repo_url = "https://github.com/AnnaTJeby/nlp_casestudy"
repo_name = repo_url.rstrip("/").split("/")[-1]

# Use the LOCAL repository if it is already present.
# Otherwise clone it into the current working directory.
current_dir = Path.cwd()
local_repo = current_dir / repo_name

if (current_dir / "train_split.csv").exists():
    repo_path = current_dir
elif local_repo.exists():
    repo_path = local_repo
else:
    import subprocess
    subprocess.run(["git", "clone", repo_url, str(local_repo)], check=True)
    repo_path = local_repo

repo_path = repo_path.resolve()
print(f"Repository path: {repo_path}")



### Load the dataset from the cloned repository

The dataset is **not loaded from `/content/` or from the notebook's assumed working directory**.  
Instead, the code below searches the local cloned repository for `train_split.csv` and `test_split.csv`.

This makes the notebook work with the actual local path of the dataset contained in the cloned repository.


In [ ]:

# Find the dataset files inside the LOCAL cloned repository
train_files = list(repo_path.rglob("train_split.csv"))
test_files = list(repo_path.rglob("test_split.csv"))

if not train_files:
    raise FileNotFoundError(f"train_split.csv was not found inside: {repo_path}")

if not test_files:
    raise FileNotFoundError(f"test_split.csv was not found inside: {repo_path}")

train_path = train_files[0]
test_path = test_files[0]

print(f"Training dataset: {train_path}")
print(f"Testing dataset:  {test_path}")

# Load the datasets using their actual local paths
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("\n--- Data Loaded Successfully ---")
print(f"Training shape: {train_df.shape}")
print(f"Testing shape:  {test_df.shape}")

display(train_df.head())
display(test_df.head())



### Inspect the columns

Before training Logistic Regression, verify that the repository datasets contain the expected text and target columns.


In [ ]:

print("Training columns:")
print(train_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())

required_columns = {"clean_text", "label"}
missing_train = required_columns - set(train_df.columns)
missing_test = required_columns - set(test_df.columns)

if missing_train:
    raise KeyError(f"Missing columns in training data: {sorted(missing_train)}")

if missing_test:
    raise KeyError(f"Missing columns in testing data: {sorted(missing_test)}")

print("\nRequired columns found: clean_text and label")
print("\nLabel distribution in training data:")
print(train_df["label"].value_counts())



## Feature Extraction and Logistic Regression with Varying `max_features`

Bag-of-Words (BoW) features are extracted from `clean_text` using `CountVectorizer`.  
A Logistic Regression classifier is then trained for three vocabulary sizes: **40,000, 5,000, and 1,000 features**.

The test set is transformed using the vectorizer fitted on the training set, which prevents information from the test set from leaking into training.


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np

# Text and target columns from the actual repository dataset
X_train_text = train_df["clean_text"].fillna("")
X_test_text = test_df["clean_text"].fillna("")
y_train = train_df["label"]
y_test = test_df["label"]

max_features_list = [40000, 5000, 1000]
results = {}

for mf in max_features_list:
    print(f"\n{'=' * 70}")
    print(f"Processing max_features = {mf}")
    print(f"{'=' * 70}")

    # 1. Fit the vectorizer ONLY on the training text
    vectorizer = CountVectorizer(max_features=mf)
    X_train_bow = vectorizer.fit_transform(X_train_text)

    # 2. Use the same fitted vocabulary to transform the test text
    X_test_bow = vectorizer.transform(X_test_text)

    print(f"Train BoW shape: {X_train_bow.shape}")
    print(f"Test BoW shape:  {X_test_bow.shape}")
    print(f"Actual vocabulary size: {len(vectorizer.vocabulary_)}")

    # 3. Train Logistic Regression
    model = LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver="liblinear"
    )
    model.fit(X_train_bow, y_train)

    # 4. Predict on the test set
    y_pred = model.predict(X_test_bow)

    # 5. Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )

    results[mf] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

    print("\n--- Model Evaluation ---")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)

    print("Confusion Matrix:")
    print(cm)
    print(f"True labels:      {np.unique(y_test).tolist()}")
    print(f"Model classes:    {model.classes_.tolist()}")

print("\n" + "=" * 70)
print("SUMMARY OF RESULTS")
print("=" * 70)

for mf, metrics in results.items():
    print(f"\nMax Features = {mf}")
    for metric, value in metrics.items():
        print(f"  {metric.replace('_', ' ').title()}: {value:.4f}")
